## EU REED

#### 1) Import packages, background databases and set project

In [1]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp

import matplotlib.pyplot as plt
import d3graph
import polyviz

17:00:32+0100 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
config = Config()

In [3]:
bd.projects.set_current(config.PROJECT_NAME)

In [4]:
import import_db

#### 2) Import foreground database 

In [5]:
bonsai = bd.Database('bonsai_231231')
bonsai_bio = bd.Database('bonsai_231231 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')

In [6]:
bd.databases

Databases dictionary with 4 object(s):
	bonsai_231231
	bonsai_231231 biosphere
	ecoinvent-3.10-biosphere
	reed

In [7]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'model'/'REED_model_20260210.xlsx',)
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])

Extracted 4 worksheets in 0.53 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [8]:
importer.statistics()

Graph statistics for `reed` importer:
32 graph nodes:
	None: 32
194 graph edges:
	technosphere: 160
	production: 32
	biosphere: 2
194 edges to the following databases:
	bonsai_231231: 110
	reed: 82
	ecoinvent-3.10-biosphere: 2
0 unique unlinked edges (0 total):




(32, 194, 0, 0)

In [9]:
importer.write_database()

100%|██████████| 32/32 [00:00<00:00, 559.99it/s]


17:00:38+0100 [info     ] Vacuuming database            
Created database: reed


In [10]:
pd.DataFrame(importer.unlinked)

""


In [11]:
reed = bd.Database('reed')
bonsai = bd.Database('bonsai_231231')

In [12]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [13]:
from IPython.display import display
display(df_reed[['code', 'comment', 'location', 'unit', 'database']])

,code,comment,location,unit,database
0,turning,proxy for turning. Water and other liquids for...,GB,kilogram,reed
1,Komatsu piece,None,GB,unit,reed
2,"soraluce machine, Goimek",Soraluce BoM for Goimek,ES,unit,reed
3,"soraluce machine, Komatsu","From Soraluce BoM, transport from ES to GB mis...",GB,unit,reed
4,"drilling, soraluce, kilogram",proxy for drilling. There are multiple differe...,GB,kilogram,reed
5,steel production,proxy for steel production,CN,kilogram,reed
6,"milling, soraluce, kilogram",proxy for milling. Waiting for primary electri...,GB,kilogram,reed
7,sandblasting,0.0266 kg direct CO2 emissions missing due to ...,DE,m2,reed
8,"flamecutting, natural gas, meter",This process represents flamecutting. The data...,DE,meter,reed
9,WWR piece,waste treatment to be added,DE,unit,reed


In [14]:
bd.databases

Databases dictionary with 4 object(s):
	bonsai_231231
	bonsai_231231 biosphere
	ecoinvent-3.10-biosphere
	reed

#### 3) Importing sensordata
###### This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [15]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [16]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [17]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    monitoring_data_path,
    tool_descriptions_path,
)
print(sensordata)

                                          energy_consumption  Power_Active  \
company event_id             Tool_Number                                     
goimek  OF10001_G_BQC_S8CF2G 0.0                    8.154403     11.789498   
                             1.0                   24.300319     11.436939   
                             2.0                   10.289347     14.996619   
                             3.0                   14.673875     15.127706   
                             4.0                    0.930889     11.800000   
...                                                      ...           ...   
wwr     OF10004_G_TWB_7SZ421 2694.0              4718.325664  13096.354965   
                             2695.0               959.320473   9700.993548   
                             2696.0              1756.356816  13036.875333   
                             2722.0               351.314922  10199.465472   
                             3350.0             28001.024411  13

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [18]:
print(ambiguous_company_tools)

   company  Tool_Number                   op_type_candidates
0   goimek            2                 [AUXILIARY, MILLING]
1   goimek            6                 [AUXILIARY, MILLING]
2   goimek           30                 [AUXILIARY, MILLING]
3   goimek           31                   [CHAMFER, MILLING]
4   goimek           44                 [AUXILIARY, MILLING]
5   goimek           45                 [AUXILIARY, MILLING]
6  komatsu            3           [AUXILIARY, ROUGH MILLING]
7  komatsu           48  [AUXILIARY, MILLING, ROUGH MILLING]
8  komatsu           50        [ROUGH BORING, ROUGH MILLING]
9  komatsu           72      [FINISH MILLING, ROUGH MILLING]


In [19]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)

In [20]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)

#### 4) Switching in sensordata

this section is missing a part about how to handle the processes that are not in hours?

In [22]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    build_consumer_input_scenarios_from_hours,
    describe_scenarios,
    lcia_for_activity_with_scenario,
    lcia_for_activity_baseline,
)

Start by selected the company and event_id where you want to swap in the sensordata

In [ ]:
#Split sensordata by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[("komatsu", "OF10006_G_SLG_E8AJ1C")]
)
print(tool_energy_by_company_and_event)

                                                          energy_consumption  \
company event_id             reed tool                                         
komatsu OF10006_G_SLG_E8AJ1C generic machining, soraluce           79.911755   
                             milling, soraluce                     25.856973   

                                                              hours  \
company event_id             reed tool                                
komatsu OF10006_G_SLG_E8AJ1C generic machining, soraluce  23.718056   
                             milling, soraluce             6.878611   

                                                          Power_Active  
company event_id             reed tool                                  
komatsu OF10006_G_SLG_E8AJ1C generic machining, soraluce      3.369237  
                             milling, soraluce                3.759040  


Find all processes present in the sensordata in the reed-database

In [23]:
tools = tool_energy_by_company_and_event.index.get_level_values("reed tool").unique()
found = {t: find_activities_by_name(reed, t, exact=False) for t in tools}
found

{'generic machining, soraluce': ['generic machining, soraluce' (hour, GB, None),
  'generic machining, soraluce' (kilogram, GB, None)],
 'milling, soraluce': ['milling, soraluce' (kilogram, GB, None),
  'milling, soraluce' (hour, GB, None)]}

In [24]:
#Identifying the electricity activity in bonsai to be able to compute the original electricity input amount for the activities found above
electricity = bonsai.get(name='electricity', location='GB', unit='TJ')

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [25]:
scenarios = build_scenarios_from_power_active(tool_energy_by_company_and_event, found, electricity)


The input of products in found to consuming activities are found and used to build scenarios.

In [26]:
consumer_scenarios = build_consumer_input_scenarios_from_hours(tool_energy_by_company_and_event, found, reed)

In [27]:
full_scenario = pd.concat([scenarios, consumer_scenarios])

In [28]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,komatsu_OF10006_G_SLG_E8AJ1C
0,electricity,TJ,"generic machining, soraluce",hour,0.000015,0.000012
1,electricity,TJ,"milling, soraluce",hour,0.000015,0.000014
2,"generic machining, soraluce",hour,Komatsu piece,unit,0.000000,23.718056
3,"milling, soraluce",hour,Komatsu piece,unit,0.000000,6.878611


Calculate a LCA score for a specific activity and scenario

In [29]:
score, lca = lcia_for_activity_with_scenario(
    activity=reed.get(name="Komatsu piece"),
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col=None,     # or your company_event column name
    amount=1
)
score


6494.502786726914

#### 5) Select processes to be analysed

In [ ]:
reed.search('welding')

In [ ]:
bonsai.search('basic iron and steel gb')

In [ ]:
bonsai_bio.search('carbon dioxide')

In [ ]:
check_activity_type = print(lpg["activity type"])

In [ ]:
machining_processes = {
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='meter'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='meter'),
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='hour'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='hour'),
    "bending": reed.get(name='bending', location="DE", unit='hour'),
    "sawing": reed.get(name='sawing', location="DE", unit='hour'),
    "welding, manual": reed.get(name='welding, manual', location="GB", unit='hour'),
    "welding, robotic": reed.get(name='welding, robotic', location="GB", unit='hour'),
    "turning": reed.get(name='turning', location="GB", unit='kilogram'),
    "painting": reed.get(name='painting', location="DE", unit='m2'),
    "Casting": reed.get(name='Casting', location="DE", unit='tonne'),
    "sandblasting": reed.get(name='sandblasting', location="DE", unit='m2'),
}

In [ ]:
soraluce_machining = {
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='kilogram'),
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='hour'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='kilogram'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='hour'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='kilogram'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='hour'),
}

In [ ]:
use_case_machines = {
    "use_case_komatsu": reed.get(name='Komatsu piece', location="GB", unit='unit'),
    "use_case_wwr": reed.get(name='WWR piece', location="DE", unit='unit'),
    "use_case_goimek": reed.get(name='Goimek piece', location="ES", unit='unit'),
}


In [ ]:
soraluce_machines = {
"soraluce_komatsu": reed.get(name='soraluce machine, Komatsu', location="GB", unit='unit'),
"soraluce_wwr": reed.get(name='soraluce machine, WWR', location="DE", unit='unit'),  
"soraluce_goimek": reed.get(name='soraluce machine, Goimek', location="ES", unit='unit'),
"Soraluce, average": reed.get(name='Soraluce, average', location="GB", unit='unit'),
}

In [ ]:
# export the LCI to Microsoft Excel in an overview format
bi.export.excel.write_lci_excel(reed.name, objs=[steel], dirpath=Path.cwd())

#### 4) Defining LCIA functions

In [ ]:
def lca_score(process_dict, method):
    results = []
    for name, process in process_dict.items():
        lca = process.lca(method)
        results.append({
            "Process": name,
            "Location": process.get('location'),
            "Functional Unit": process.get('unit'),
            "Climate Impact (kg CO2eq)": lca.score
        })

    return pd.DataFrame(results)

In [ ]:
def run_contribution_analysis(demand_object,
                              lcia_method,
                              max_level=1,
                              cutoff=0.01):
    return bwa.utils.recursive_calculation_to_object(
        demand_object,
        lcia_method=lcia_method,
        max_level=max_level,
        cutoff=cutoff,
        as_dataframe=True,
    )

In [ ]:
def plot_contribution(df,
                      title='Contribution Analysis',
                      xlabel='CO2 eq. emissions (kg)',
                      drop_parent=True):
    df_plot = df.copy()

    # Drop the parent (the demand itself)
    if drop_parent:
        # Heuristic: parent is usually the one with maximum score
        parent_score = df_plot['score'].max()
        df_plot = df_plot[df_plot['score'] < parent_score]

    # Sort descending by contribution
    df_plot = df_plot.sort_values(by='score', ascending=False)

    f, ax = plt.subplots(figsize=(5, 2))
    sns.barplot(y='name', x='score', data=df_plot, ax=ax)

    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=''
    )
    plt.tight_layout()
    plt.show()

#### 5) Calculate and present results

In [ ]:
df_results = lca_score(machining_processes, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
flamecutting = reed.get(name = 'flamecutting, lpg', location = "DE", unit = 'hour')

In [ ]:
# Another option is to use the recursive calculation to an object, which returns a DataFrame
flamecutting_ca = bwa.utils.recursive_calculation_to_object(flamecutting,
                                          lcia_method=config.lcia_gwp100,
                                          max_level=3,
                                          cutoff=0.001,
                                          as_dataframe=True,
                                          )
flamecutting_ca

In [ ]:
df_results = lca_score(soraluce_machining, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_results = lca_score(use_case_machines, config.lcia_gwp100)
display(df_results)

In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

#### 6) Run contribution analysis

In [ ]:
processes_results_ca = {}

for key, demand in soraluce_machining.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    processes_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
use_case_results_ca = {}

for key, demand in use_case_machines.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    use_case_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )